# Tutorial 04: Multi-Agent Workflow Orchestration (Offline-First)

Deep dive into `A2A/04_multi_agent_workflow` where planning, delegation, invocation, and result chaining are executed as one workflow.

## 1) Where We Are in the Journey

`03_agent_invocation` completed single-hop execution: choose one agent and call one endpoint.
This step exists to orchestrate multiple dependent steps in one query and pass intermediate outputs forward.
It is the first true workflow layer.

## 2) What You Will Learn

- How `plan(query)` decomposes a compound request into ordered steps
- How workflow state carries outputs across steps
- How routing-by-tool and invocation-by-endpoint interact
- Why chain-of-tools orchestration differs from single-call invocation

## 3) Why This Matters

Production agent systems usually need multiple operations, not one.
Workflow orchestration makes cross-agent collaboration explicit and testable.
This is where A2A planning and MCP-style execution start working together at system level.

## 4) Core Concept Deep Dive

Intuition: a query like "calculate interest ... then add ..." is a pipeline, not a single tool call.
System design: `discover_capabilities` -> `plan` -> loop(`build payload`, `invoke`, `update state`).
Trade-off: simple planning is transparent but brittle to natural-language variation.
Key detail from this folder: the second step injects prior `result` as input `a` for `add`.

## 5) Code Walkthrough (Only `A2A/04_multi_agent_workflow`)

- `discover_capabilities()` builds registry from `/agent-info` and `/tools`.
- `plan(query)` parses numbers and creates ordered finance -> math steps.
- `invoke(agent, registry, payload)` resolves endpoint and executes request.
- `execute(query)` runs the loop, injects previous result, and returns final output.
Offline adaptation here preserves exact step and payload shapes while replacing network calls.

In [ ]:
import re

# Schema-faithful offline registry matching 00_Agents contracts
MOCK_REGISTRY = [
    {
        'agent': 'finance-agent',
        'endpoint': 'http://localhost:8001/invoke',
        'tools': [{'name': 'calculate_interest', 'description': 'Calculate simple interest'}]
    },
    {
        'agent': 'math-agent',
        'endpoint': 'http://localhost:8000/invoke',
        'tools': [{'name': 'add', 'description': 'Add two numbers'}]
    }
]

In [ ]:
def discover_capabilities():
    # Offline-first mirror of discovery output
    return MOCK_REGISTRY

def plan(query):
    query = query.lower()
    numbers = list(map(float, re.findall(r'\d+\.?\d*', query)))
    steps = []
    if 'interest' in query:
        steps.append({'agent': 'finance-agent', 'tool': 'calculate_interest', 'args': {'principal': numbers[0], 'rate': numbers[1], 'time': numbers[2]}})
    if 'add' in query:
        steps.append({'agent': 'math-agent', 'tool': 'add', 'args': {'b': numbers[-1]}})
    return steps

In [ ]:
def invoke(agent, registry, payload):
    # Offline simulator of MCP invocation
    if agent == 'finance-agent' and payload['tool'] == 'calculate_interest':
        a = payload['args']
        return {'status': 'success', 'result': (a['principal'] * a['rate'] * a['time']) / 100}
    if agent == 'math-agent' and payload['tool'] == 'add':
        a = payload['args']
        return {'status': 'success', 'result': a['a'] + a['b']}
    return {'status': 'error', 'message': 'Unsupported call'}

def execute(query):
    registry = discover_capabilities()
    steps = plan(query)
    state = {'result': None}
    print('PLAN:', steps)
    for i, step in enumerate(steps, 1):
        payload = {'tool': step['tool'], 'args': dict(step['args'])}
        if payload['tool'] == 'add':
            payload['args']['a'] = state['result']
        print(f'STEP {i}: {step[
]} -> {payload}')
        response = invoke(step['agent'], registry, payload)
        state['result'] = response.get('result')
        print('RESULT:', response)
    return state['result']

In [ ]:
query = 'calculate interest for 2000 at 10 for 1 year and then add 500'
final = execute(query)
print('FINAL:', final)

## 6) System Flow Explanation

1. Discover available capabilities.
2. Convert query into ordered steps.
3. Route each step to matching agent.
4. Invoke tool with structured payload.
5. Write result into state and feed next step.
6. Return final composed output.

## 7) Important Concepts You Might Miss

- Workflow state is a first-class artifact.
- Step order is semantic, not cosmetic.
- Payload injection (`a = prior result`) is the chaining mechanism.
- Planning errors propagate into every downstream step.

## 8) Common Mistakes / Pitfalls

- Building steps without enough numbers in query.
- Forgetting to inject prior result for chained math step.
- Treating workflow as parallel when it is dependency-ordered.
- Assuming every step succeeds without validation.

## 9) Key Takeaways

- This folder introduces true multi-step orchestration.
- State and payload shaping are central design concerns.
- Offline simulation can validate orchestration semantics before deployment.

## 10) What’s Next (Strict Preview)

`05_Negotiation` changes strategy from single planned path to multi-agent competition/cooperation.
It will solve how to collect multiple candidate responses and select the best one after execution.
This matters when uncertainty is high and one-shot routing is risky.